# Training the ML Engine on Google Colab

**IMPORTANT**: Use a **GPU runtime** (T4 is free). In Colab: `Runtime → Change runtime type → T4 GPU`.
Training on CPU in Colab is ~10× slower.

---

## Step 0 — Upload Your `ml_engine/` Scripts
You need the four Python scripts from `ml_engine/` in Colab.

### Option A — Mount Google Drive (recommended)
Put the scripts in Drive once, reuse across sessions.

In [ ]:
# Cell 0a
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree('/content/drive/MyDrive/blacksite/ml_engine', '/content/ml_engine', dirs_exist_ok=True)

### Option B — Upload directly (no Drive)

In [ ]:
# Cell 0b
from google.colab import files
import os

uploaded = files.upload()  # select all 4 .py files at once

os.makedirs('/content/ml_engine', exist_ok=True)
for fname in uploaded.keys():
    os.rename(fname, f'/content/ml_engine/{fname}')

---
## Step 1 — Install Dependencies

In [ ]:
# Cell 1
!pip install -q "tensorflow>=2.13.0,<2.17.0" numpy onnxruntime>=1.17.0 tf2onnx>=1.15.0 onnx>=1.14.0

import tensorflow as tf
print("TF version :", tf.__version__)
print("GPUs found :", tf.config.list_physical_devices('GPU'))
# Expected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

---
## Step 2 — Download Datasets
Three complementary sources that together give the best coverage of real-world password patterns:
- **RockYou** (14.3M) — English patterns, leet speak baseline
- **SecLists 10M** (10M) — Post-2016 breaches, default credentials
- **Probable Wordlists Top95** (~30M) — Frequency-ranked real passwords, multi-language

In [ ]:
# Cell 2
import os
os.makedirs('/content/datasets', exist_ok=True)

# 1. RockYou (direct mirror, ~130 MB)
!wget -q --show-progress -O /content/datasets/rockyou.txt \
    https://github.com/brannondorsey/naive-hashcat/releases/download/data/rockyou.txt

# 2. SecLists — top 10 million passwords
!wget -q --show-progress -O /content/datasets/seclists-10m.txt \
    https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/xato-net-10-million-passwords.txt

# 3. Probable Wordlists — Top 95% probable real passwords
!wget -q --show-progress -O /content/datasets/probable-top95.txt \
    https://github.com/berzerk0/Probable-Wordlists/raw/master/Real-Passwords/Top95Probable-Real-Passwords-Top1Bytes.txt

# Verify downloads
!echo "--- Line counts ---"
!wc -l /content/datasets/*.txt

---
## Step 3 — Merge and Deduplicate
Training on duplicates wastes GPU cycles and skews the model toward over-represented passwords. This cell merges all sources, deduplicates, and shuffles before writing.

In [ ]:
# Cell 3
import random

INPUT_FILES = [
    '/content/datasets/rockyou.txt',
    '/content/datasets/seclists-10m.txt',
    '/content/datasets/probable-top95.txt',
]

OUTPUT_FILE = '/content/datasets/combined_passwords.txt'
MAX_TOTAL   = 3_000_000   # safe for Colab's ~12 GB RAM; raise to 5M on Colab Pro

seen      = set()
passwords = []

for filepath in INPUT_FILES:
    before = len(passwords)
    try:
        with open(filepath, 'r', encoding='latin-1', errors='replace') as fh:
            for line in fh:
                pw = line.rstrip('\n\r')
                if 4 <= len(pw) <= 64 and pw not in seen:
                    seen.add(pw)
                    passwords.append(pw)
                    if len(passwords) >= MAX_TOTAL:
                        break
        added = len(passwords) - before
        print(f"  +{added:>9,} from {filepath.split('/')[-1]}  →  {len(passwords):,} total unique")
    except FileNotFoundError:
        print(f"  SKIPPED (not found): {filepath}")
    if len(passwords) >= MAX_TOTAL:
        break

# Shuffle to prevent training on one source at a time
random.shuffle(passwords)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(passwords))

print(f"\n✓ Combined file: {len(passwords):,} unique passwords → {OUTPUT_FILE}")
del seen   # free RAM before training

---
## Step 4 — Prepare the Dataset

In [ ]:
# Cell 4
!python /content/ml_engine/01_data_preparation.py \
    --passwords /content/datasets/combined_passwords.txt \
    --out_dir   /content/data \
    --seq_len   10 \
    --max_rows  3000000

# Expected output:
#   [DataPrep] Vocab built: 97 tokens
#   [DataPrep] Done. Accepted: ~2 950 000 | Skipped: ~50 000
#   [DataPrep] Dataset shape → X: (~25 000 000, 10), y: (~25 000 000, 97)

---
## Step 5 — Train the Model

In [ ]:
# Cell 5b — run BEFORE Cell 5
%load_ext tensorboard
%tensorboard --logdir /content/models/logs

In [ ]:
# Cell 5
!python /content/ml_engine/02_train.py \
    --data_dir   /content/data \
    --model_dir  /content/models \
    --epochs     20 \
    --batch      2048 \
    --embed_dim  32 \
    --lstm_units 128

# batch=2048 saturates the T4 and is ~4× faster per epoch than batch=512.
# Expected time: ~10–15 min total for 3M passwords, 20 epochs on T4.

---
## Step 6 — Export to ONNX (and TFLite fallbacks)

In [ ]:
# Cell 6
!python /content/ml_engine/03_export.py \
    --model_dir     /content/models \
    --data_dir      /content/data \
    --export_dir    /content/exports

# Expected output:
#   [Export] ✓ ONNX → /content/exports/password_model.onnx  (~396 KB)
#   [Export] ✓ F16 TFLite → /content/exports/password_model_f16.tflite  (~207 KB)
#   [Export] ✓ Dynamic INT8 TFLite → /content/exports/password_model_dynamic.tflite (~137 KB)
#   [Export] ✓ ONNX verification passed.

---
## Step 7 — Benchmark and Calibrate Thresholds

In [ ]:
# Cell 7
!python /content/ml_engine/inference.py \
    --model  /content/exports/password_model.onnx \
    --vocab  /content/exports/vocab.json \
    --meta   /content/exports/dataset_meta.json \
    --benchmark

# Inspect the NLL column carefully:
#   "password", "123456"         → should be < 1.5  (Weak)
#   "P@ssw0rd!", "Hunter2"       → should be 1.5–3.0 (Moderate)
#   "xK#9mP2$vQzL"               → should be > 3.0  (Strong)
#
# If the tiers don't look right, adjust STRENGTH_TIERS in inference.py
# (around line 60) and re-upload the file.

---
## Step 8 — Save and Download

In [ ]:
# Cell 8a
import shutil
shutil.copytree('/content/exports', '/content/drive/MyDrive/blacksite/exports',
                dirs_exist_ok=True)
print("✓ Exports saved to Google Drive")

In [ ]:
# Cell 8b
import shutil
from google.colab import files

shutil.make_archive('/content/ml_engine_exports', 'zip', '/content/exports')
files.download('/content/ml_engine_exports.zip')

# Unzip locally and place contents in:
#   d:\MM\0dev\Blacksite_Node\ml_engine\exports\
#     ├── password_model.onnx          (~396 KB)
#     ├── password_model_f16.tflite    (~207 KB)
#     ├── password_model_dynamic.tflite(~137 KB)
#     ├── vocab.json
#     └── dataset_meta.json